# 10 — Testing & Design Patterns
**References:** Beazley *Python Cookbook* Ch. 14 · pytest docs · Percival *Architecture Patterns with Python* · Gamma et al. *Design Patterns*

## Narrative thread
```
pytest basics -> Fixtures -> Parametrize -> Mocking -> Property-based testing -> Design patterns
```

## 1. pytest: the test framework

pytest discovers tests in any file named `test_*.py` or `*_test.py`. Functions named `test_*` are test cases. No boilerplate classes needed.

**Key advantages over `unittest`:**
- Plain `assert` statements (pytest rewrites them for rich failure messages)
- Fixtures over `setUp`/`tearDown`
- Parametrize over test loops
- Rich plugin ecosystem (`pytest-cov`, `pytest-asyncio`, `pytest-xdist`)

In [ ]:
# This cell shows code that would be in test files.
# In a real project, run: uv run pytest -v

import pytest
from unittest.mock import MagicMock, patch, AsyncMock, call
from dataclasses import dataclass
from typing import Protocol

# ── System under test ─────────────────────────────────────────────────────
@dataclass
class User:
    id:    int
    name:  str
    email: str

class UserRepository(Protocol):
    def get(self, user_id: int) -> User: ...
    def save(self, user: User) -> None: ...

class UserService:
    def __init__(self, repo: UserRepository):
        self._repo = repo

    def get_display_name(self, user_id: int) -> str:
        user = self._repo.get(user_id)
        return f'{user.name} <{user.email}>'

    def update_email(self, user_id: int, new_email: str) -> None:
        if '@' not in new_email:
            raise ValueError(f'Invalid email: {new_email}')
        user = self._repo.get(user_id)
        user.email = new_email
        self._repo.save(user)

# ── Fixtures ──────────────────────────────────────────────────────────────
# In a real test file:
# @pytest.fixture
# def mock_repo():
#     repo = MagicMock(spec=UserRepository)
#     repo.get.return_value = User(1, 'Alice', 'alice@example.com')
#     return repo
#
# @pytest.fixture
# def service(mock_repo):
#     return UserService(mock_repo)

# ── Demonstrate tests running directly ───────────────────────────────────
mock_repo = MagicMock(spec=['get', 'save'])
mock_repo.get.return_value = User(1, 'Alice', 'alice@example.com')

service = UserService(mock_repo)

# Test: get_display_name
result = service.get_display_name(1)
assert result == 'Alice <alice@example.com>'
mock_repo.get.assert_called_once_with(1)
print(f'get_display_name: {result!r} ✓')

# Test: update_email — happy path
mock_repo.reset_mock()
service.update_email(1, 'newalice@example.com')
assert mock_repo.get.called
assert mock_repo.save.called
saved_user = mock_repo.save.call_args[0][0]
assert saved_user.email == 'newalice@example.com'
print('update_email happy path ✓')

# Test: update_email — invalid email
try:
    service.update_email(1, 'not-an-email')
    assert False, 'Should have raised'
except ValueError:
    print('update_email invalid email raises ValueError ✓')

In [ ]:
# ── Parametrize: test many inputs concisely ───────────────────────────────
# In real pytest:
# @pytest.mark.parametrize('email,valid', [
#     ('alice@example.com', True),
#     ('not-an-email',       False),
#     ('a@b',                True),
#     ('',                   False),
# ])
# def test_email_validation(email, valid):
#     ...

# Simulate parametrize manually for notebook
test_cases = [
    ('alice@example.com', True),
    ('not-an-email',       False),
    ('a@b',                True),
    ('',                   False),
    ('x@y.z',              True),
]

def is_valid_email(email: str) -> bool:
    return '@' in email and len(email) > 0

print('Parametrized email tests:')
for email, expected in test_cases:
    result = is_valid_email(email)
    status = '✓' if result == expected else '✗'
    print(f'  {status} is_valid_email({email!r}) == {expected}')

print()
# ── patch: mock specific imports ──────────────────────────────────────────
import datetime

def get_greeting(name: str) -> str:
    hour = datetime.datetime.now().hour
    if hour < 12:   return f'Good morning, {name}!'
    elif hour < 17: return f'Good afternoon, {name}!'
    else:           return f'Good evening, {name}!'

# Patch datetime to control what 'now' returns
with patch('datetime.datetime') as mock_dt:
    mock_dt.now.return_value = datetime.datetime(2024, 1, 1, 9, 0)  # 9am
    assert get_greeting('Alice') == 'Good morning, Alice!'
    print('Morning greeting ✓')

    mock_dt.now.return_value = datetime.datetime(2024, 1, 1, 14, 0)  # 2pm
    assert get_greeting('Bob') == 'Good afternoon, Bob!'
    print('Afternoon greeting ✓')

## 2. Design Patterns in Python

Many GoF patterns are simpler in Python because of first-class functions, duck typing, and the data model.

In [ ]:
from __future__ import annotations
from typing import Callable
from abc import ABC, abstractmethod

# ── Strategy: replace with first-class functions ──────────────────────────
# Classic OOP Strategy uses ABC + subclasses.
# In Python, the strategy IS a function.

SortStrategy = Callable[[list], list]

class Sorter:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    def sort(self, data: list) -> list:
        return self._strategy(data)

data = [3, 1, 4, 1, 5, 9, 2, 6]
print('Strategy pattern:')
print(f'  Ascending:  {Sorter(sorted).sort(data)}')
print(f'  Descending: {Sorter(lambda x: sorted(x, reverse=True)).sort(data)}')
print(f'  By length:  {Sorter(lambda x: sorted(str(i) for i in x)).sort(data)}')

print()
# ── Observer: event system ─────────────────────────────────────────────────
class EventEmitter:
    def __init__(self):
        self._handlers: dict[str, list[Callable]] = {}

    def on(self, event: str) -> Callable:
        """Decorator: @emitter.on('click') def handler(data): ..."""
        def decorator(fn: Callable) -> Callable:
            self._handlers.setdefault(event, []).append(fn)
            return fn
        return decorator

    def emit(self, event: str, **data) -> None:
        for handler in self._handlers.get(event, []):
            handler(**data)

events = EventEmitter()

@events.on('user_created')
def send_welcome_email(name, email):
    print(f'  [email] Welcome email to {name} <{email}>')

@events.on('user_created')
def log_creation(name, email):
    print(f'  [log]   User created: {name}')

@events.on('user_deleted')
def cleanup(name, **_):
    print(f'  [cleanup] Removing data for {name}')

print('Observer pattern:')
events.emit('user_created', name='Alice', email='alice@example.com')
events.emit('user_deleted', name='Bob')

print()
# ── Builder: fluent API ───────────────────────────────────────────────────
class Query:
    def __init__(self, table: str):
        self._table = table
        self._conditions: list[str] = []
        self._order:  str | None = None
        self._limit:  int | None = None
        self._columns: list[str] = ['*']

    def select(self, *cols: str) -> Query:
        self._columns = list(cols); return self

    def where(self, condition: str) -> Query:
        self._conditions.append(condition); return self

    def order_by(self, col: str, desc: bool = False) -> Query:
        self._order = f'{col} DESC' if desc else col; return self

    def limit(self, n: int) -> Query:
        self._limit = n; return self

    def build(self) -> str:
        sql  = f'SELECT {', '.join(self._columns)} FROM {self._table}'
        if self._conditions:
            sql += ' WHERE ' + ' AND '.join(self._conditions)
        if self._order:
            sql += f' ORDER BY {self._order}'
        if self._limit:
            sql += f' LIMIT {self._limit}'
        return sql

print('Builder pattern:')
sql = (
    Query('users')
    .select('id', 'name', 'email')
    .where('age > 18')
    .where('active = 1')
    .order_by('created_at', desc=True)
    .limit(10)
    .build()
)
print(f'  {sql}')

## 3. Key takeaways

| Concept | Rule |
|---|---|
| `pytest` fixtures | Dependency injection for tests; scope controls lifetime |
| `@pytest.mark.parametrize` | Test many inputs without loops |
| `MagicMock(spec=...)` | Spec limits to real attributes — prevents typos in tests |
| `patch` | Context manager or decorator to replace imports during tests |
| Strategy pattern | In Python: just pass a function; no ABC needed |
| Observer pattern | `dict[str, list[Callable]]` — simpler than class hierarchy |
| Builder pattern | Fluent `return self` API for constructing complex objects |

**Next:** notebook 11 — packaging, tooling, and `pyproject.toml`.